In [150]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model = load_model("../../models/model_2_1/model_2_1.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)


In [151]:
# Coger los 100 primeros registros para hacer predicciones de ejemplo
stations_start = df_data["start_station_idx"]
stations_end = df_data["end_station_idx"]
context = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual",
    "start_station_idx",
    "end_station_idx"
])

In [152]:
num_records = 100

In [153]:
stations_start_subset = stations_start.iloc[:num_records].values
stations_end_subset = stations_end.iloc[:num_records].values
context_subset = context.iloc[:num_records].values.astype(np.float32)

In [154]:
station_offset = 0
num_features = context.shape[1]

print(num_features)

27


In [155]:
results = []

In [156]:
# ---------- Bucle de predicción
for real_start, item_context, real_end in zip(stations_start_subset, context_subset, stations_end_subset):
    # Preparar inputs
    real_context_input = item_context.reshape(1, context_subset.shape[1])
    
    start_station_idx = real_start - station_offset
    real_start_input = np.array([start_station_idx]).reshape(1, 1)
    
    end_station_idx = real_end - station_offset
    real_end_input = np.array([end_station_idx]).reshape(1, 1)    
    
    # Dummy para end_station_input
    dummy_end_input = np.zeros((1, 1))

    # Hacer predicción    
    predictions = model.predict({
        'context': real_context_input,
        'start_station_input': real_start_input,
        'end_station_input': real_end_input
    }, verbose=0)
    
    predicted_end_station_idx = np.argmax(predictions, axis=1)[0] + station_offset
    predicted_probability = np.max(predictions, axis=1)[0]

    # Guardar resultados en la lista
    results.append({
        "start_station_idx": real_start,
        "end_station_idx": real_end,
        "predicted_end_station_idx": predicted_end_station_idx,
        "predicted_probability": predicted_probability
    })
    
df_results = pd.DataFrame(results)
print(df_results.head())    

   start_station_idx  end_station_idx  predicted_end_station_idx  \
0               1281             1896                       1899   
1               1819             1645                       1645   
2               1250             1250                       1640   
3               1896             1911                          2   
4               1893             1646                       1380   

   predicted_probability  
0               0.029551  
1               0.949145  
2               0.032974  
3               0.017139  
4               0.041566  


In [157]:
df_results = df_results[df_results["predicted_probability"] >= 0.6]

In [158]:
df_results["fail"] = df_results["end_station_idx"] != df_results["predicted_end_station_idx"]

In [162]:
df_good = df_results[~df_results["fail"]]

In [164]:
df_good.shape

(14, 5)

# --------------------------------------------------------------------------------------------------------

In [169]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model = load_model("../../models/model_2_1/model_2_1.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

# Quitar columnas que no son features
df_model_feature = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual",
    "start_station_idx",
    "end_station_idx"
])

# ---------- Configuración necesaria para reconstruir índices
station_offset = 0
num_features = df_model_feature.shape[1]
num_end_stations = df_data["end_station_idx"].max() + 1  # cantidad de estaciones finales

print("Número de features:", num_features)
print("Número de estaciones finales:", num_end_stations)

# ---------- Registro a predecir (ejemplo)
real_record = df_model_feature.iloc[0].values.astype(np.float32)
real_context_input = real_record.reshape(1, num_features)

# Estación de inicio real
start_station_idx = df_data["start_station_idx"].iloc[0] - station_offset
real_start_input = np.array([start_station_idx]).reshape(1, 1)

# ---------- Buscar la estación final con mejor predicción
best_prob = -1
best_station_id = None

for end_idx in range(num_end_stations):
    end_input = np.array([end_idx]).reshape(1, 1)
    
    pred = model.predict({
        'context': real_context_input,
        'start_station_input': real_start_input,
        'end_station_input': end_input
    }, verbose=0)
    
    prob = np.max(pred)  # probabilidad máxima para esta opción
    if prob > best_prob:
        best_prob = prob
        best_station_id = end_idx + station_offset

print("Estación de inicio real:", start_station_idx + station_offset)
print("Estación de fin real:", df_data["end_station_idx"].iloc[0])
print("Mejor estación predicha:", best_station_id, "con probabilidad:", best_prob)


Número de features: 27
Número de estaciones finales: 1912
Estación de inicio real: 1281
Estación de fin real: 1896
Mejor estación predicha: 5 con probabilidad: 1.0


# ------------------------------------------------------------------------------------------------

In [1]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model = load_model("../../models/model_2_1/model_2_1.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

# # Quitar columnas que no son features
# df_model = df_data.drop(columns=[
#     "ride_id",
#     "started_at",
#     "ended_at",
#     "start_station_id",
#     "end_station_id",
#     "time_hms_ms",
#     "member_casual"
# ])

df_model_feature = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual",
    "start_station_idx",
    "end_station_idx"
])


In [2]:
# ---------- Configuración necesaria para reconstruir índices
station_offset = 0
num_features = df_model_feature.shape[1]

print(num_features)

27


In [3]:
# ---------- Supongamos que tienes un registro real a predecir
# Aquí tomamos la primera fila como ejemplo, pero puede ser cualquier registro
real_record = df_model_feature.iloc[0].values.astype(np.float32)
real_context_input = real_record.reshape(1, num_features)

# Estación de inicio real (ejemplo)
start_station_id = df_data["start_station_idx"].iloc[0]  # reemplaza con tu valor real
start_station_idx = start_station_id - station_offset
real_start_input = np.array([start_station_idx]).reshape(1, 1)

# Dummy para end_station_input
dummy_end_input = np.zeros((1, 1))

In [4]:
# ---------- Predicción
pred = model.predict({
    'context': real_context_input,
    'start_station_input': real_start_input,
    'end_station_input': dummy_end_input
})

# ---------- Obtener estación predicha
pred_idx = np.argmax(pred, axis=1)[0]
pred_station_id = pred_idx + station_offset  # revertir offset
predicted_probability = np.max(pred, axis=1)[0]


print(f"Estación predicha: {pred_station_id} con probabilidad {predicted_probability}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Estación predicha: 997 con probabilidad 0.02432454377412796


In [8]:
pred

array([[7.2414684e-03, 5.0235488e-03, 2.2009987e-04, ..., 1.0705452e-04,
        7.2020424e-07, 1.5356424e-04]], shape=(1, 1912), dtype=float32)

In [19]:
real_record

array([ 2.0220000e+03,  1.0000000e+00,  4.0000000e+00,  0.0000000e+00,
       -5.4699998e+00,  3.0999999e+00,  0.0000000e+00,  7.4849998e+01,
        0.0000000e+00,  1.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        1.0000000e+00,  0.0000000e+00,  1.0000000e+00,  0.0000000e+00,
       -2.3483183e+00, -5.7336187e-01,  9.1600096e-01, -9.9704266e-02,
       -5.6148447e-02,  4.6411109e+00,  9.3738335e-01,  3.4829938e-01,
        5.0000000e-01,  8.6602539e-01,  1.3333333e+01], dtype=float32)

In [ ]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model = load_model("../../models/model_2_1/model_2_1.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

# Quitar columnas que no son features
df_model = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual"
])

df_model_feature = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual",
    "start_station_idx",
    "end_station_idx"
])

# ---------- Configuración necesaria para reconstruir índices
station_offset = 0
num_features = df_model_feature.shape[1]

# ---------- Supongamos que tienes un registro real a predecir
# Aquí tomamos la primera fila como ejemplo, pero puede ser cualquier registro
real_record = df_model.iloc[0].values.astype(np.float32)
real_context_input = real_record.reshape(1, num_features)

# Estación de inicio real (ejemplo)
start_station_id = df_data["start_station_idx"].iloc[0]  # reemplaza con tu valor real
start_station_idx = start_station_id - station_offset
real_start_input = np.array([start_station_idx]).reshape(1, 1)

# Dummy para end_station_input
dummy_end_input = np.zeros((1, 1))

# ---------- Predicción
pred = model.predict({
    'context': real_context_input,
    'start_station_input': real_start_input,
    'end_station_input': dummy_end_input
})

# ---------- Obtener estación predicha
pred_idx = np.argmax(pred, axis=1)[0]
pred_station_id = pred_idx + station_offset  # revertir offset
print("Estación predicha:", pred_station_id)


ValueError: Input 0 of layer "functional" is incompatible with the layer: expected shape=(None, 27), found shape=(1, 29)